# DPO reasoning evaluation and temperature scaling
Compare the original model and the validation-selected DPO adapter using direct answers and generated reasoning. Preserve raw metrics as the primary experiment results. Temperature scaling is a separate post-processing experiment and never changes model weights.

All execution switches default to False. GPU evaluation is optional; existing direct-answer CSV files are sufficient for CPU calibration. Checkpoints remain in Drive or the downloaded backup. No checkpoint is copied by this notebook.

In [1]:
from pathlib import Path
import sys, json, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = Path('/content/drive/MyDrive/medqa-calibration')
else:
    here = Path.cwd().resolve()
    REPO = here if (here / 'train/dpo/evaluate.py').is_file() else here.parent
assert (REPO / 'train/dpo/evaluate.py').is_file(), 'Set REPO to the repository root.'
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('Repository:', REPO)

Repository: C:\Users\RyanW\OneDrive\Desktop\CS 5787 Deep Learning\Final Project\medqa-calibration


In [2]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'train/dpo/requirements.txt')])

def run_command(command):
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            process.wait()
        raise
    finally:
        process.stdout.close()

## Configuration
The default adapter is the best validation checkpoint (epoch 1, step 208) from the first completed experiment. This choice is fixed before post-hoc test evaluation. Use the same quantization, prompts, token budget, and generation settings for both models.

CoT uses greedy generation, the DPO training prompt, and full-vocabulary letter rescoring after the reasoning. Generated-answer accuracy, malformed outputs, and truncation are also reported. It is distinct from the existing GRPO vLLM evaluation protocol; do not pool their results.

In [3]:
RUN_EVALUATION = False
RUN_CALIBRATION = True
SOURCE = 'hf' if IN_COLAB else 'local'
MODEL = 'Qwen/Qwen2.5-3B-Instruct' if SOURCE == 'hf' else str(REPO.parent / 'Qwen2.5-3B-Instruct')
RUN_NAME = 'dpo_full_3epochs_seed1_lr5e-6_beta0p1_r16_drop0p05'
RUN = REPO / 'outputs' / RUN_NAME
BEST = json.loads((RUN / 'best_validation.json').read_text())
DPO_NAME = f"step_{BEST['step']:06d}"
ADAPTER = RUN / 'best_adapter'
if not IN_COLAB and not (ADAPTER / 'adapter_model.safetensors').is_file():
    ADAPTER = (REPO.parent / '\u5907\u4efd\u6587\u4ef6\u5939' /
               'medqa-calibration-20261109T203726Z-1-001/medqa-calibration/outputs' /
               RUN_NAME / 'best_adapter')
EVAL_ROOT = REPO / 'outputs/dpo_posthoc_v1'
CALIBRATION_ROOT = REPO / 'outputs/dpo_temperature_v1'
MODES = ['direct', 'cot']
SPLITS = ['validation', 'test']
N_QUESTIONS = 0  # Zero means all 500 validation and all official test questions.
BATCH_SIZE = 2
COT_MAX_NEW_TOKENS = 512
COT_MAX_CONTEXT_TOKENS = 4096
print('Selected adapter:', ADAPTER)
print('Selected validation step:', BEST['step'])

Selected adapter: C:\Users\RyanW\OneDrive\Desktop\CS 5787 Deep Learning\Final Project\备份文件夹\medqa-calibration-20261109T203726Z-1-001\medqa-calibration\outputs\dpo_full_3epochs_seed1_lr5e-6_beta0p1_r16_drop0p05\best_adapter
Selected validation step: 208


In [4]:
if RUN_EVALUATION:
    assert (ADAPTER / 'adapter_model.safetensors').is_file(), 'Set ADAPTER to the retained adapter directory.'
    for name, adapter in [('base', None), (DPO_NAME, ADAPTER)]:
        command = [sys.executable, '-u', '-m', 'train.dpo.evaluate',
                   '--source', SOURCE, '--model', MODEL, '--model_name', name,
                   '--output_dir', str(EVAL_ROOT / name), '--quantization', '4bit',
                   '--batch_size', str(BATCH_SIZE), '--n', str(N_QUESTIONS),
                   '--cot_max_new_tokens', str(COT_MAX_NEW_TOKENS),
                   '--cot_max_context_tokens', str(COT_MAX_CONTEXT_TOKENS),
                   '--modes', *MODES, '--splits', *SPLITS]
        if adapter is not None:
            command += ['--adapter', str(adapter)]
        run_command(command)
else:
    print('GPU evaluation is disabled. Existing predictions can still be calibrated.')

GPU evaluation is disabled. Existing predictions can still be calibrated.


## Calibration data and interpretation
Fit a separate positive scalar temperature for each model and mode by minimizing NLL. A fixed seeded 250/250 split of the 500 validation predictions provides calibration-fit and diagnostic holdout subsets. Because this validation set already selected the checkpoint, the holdout is exploratory, not an independent estimate of the entire model-selection procedure. Official test predictions, when available, are evaluated with the already fitted temperature and are never used to fit or choose it.

The same validation question IDs must be used across models/modes. Raw and calibrated metrics remain separate. Old CSV files use clipped log probabilities reconstructed from their probability columns; new evaluations save log probabilities directly. This is answer-probability calibration, not generation temperature.

The default direct inputs are the synced Colab outputs. Fresh evaluation outputs take precedence. CoT is included only after its predictions exist. Missing test outputs are reported explicitly; no substitute test scores are fabricated.

In [5]:
INPUTS = {
    ('base', 'direct'): (REPO / 'outputs/base_medqa_4bit/validation/base_predictions.csv',
                         REPO / 'outputs/base_medqa_4bit/test/base_predictions.csv'),
    (DPO_NAME, 'direct'): (RUN / 'validation' / f'{DPO_NAME}_predictions.csv', None),
}
for name in ['base', DPO_NAME]:
    for mode in MODES:
        validation_csv = EVAL_ROOT / name / 'validation' / mode / f'{name}_predictions.csv'
        test_csv = EVAL_ROOT / name / 'test' / mode / f'{name}_predictions.csv'
        if validation_csv.is_file():
            INPUTS[(name, mode)] = (validation_csv, test_csv if test_csv.is_file() else None)
for key, paths in INPUTS.items():
    print(key, paths)

('base', 'direct') (WindowsPath('C:/Users/RyanW/OneDrive/Desktop/CS 5787 Deep Learning/Final Project/medqa-calibration/outputs/base_medqa_4bit/validation/base_predictions.csv'), WindowsPath('C:/Users/RyanW/OneDrive/Desktop/CS 5787 Deep Learning/Final Project/medqa-calibration/outputs/base_medqa_4bit/test/base_predictions.csv'))
('step_000208', 'direct') (WindowsPath('C:/Users/RyanW/OneDrive/Desktop/CS 5787 Deep Learning/Final Project/medqa-calibration/outputs/dpo_full_3epochs_seed1_lr5e-6_beta0p1_r16_drop0p05/validation/step_000208_predictions.csv'), None)


In [6]:
if RUN_CALIBRATION:
    import numpy as np
    import pandas as pd
    from eval.temperature import read_frame, fit_artifact, apply_artifact, report
    from train.dpo.validation import write_json
    assert not CALIBRATION_ROOT.exists(), 'Choose a new calibration output directory.'
    frames = {key: read_frame(paths[0]) for key, paths in INPUTS.items()}
    canonical = sorted(next(iter(frames.values())).question_id.astype(str))
    assert len(canonical) == 500, 'Calibration expects all 500 fixed validation questions.'
    for key, frame in frames.items():
        assert sorted(frame.question_id.astype(str)) == canonical, f'Validation IDs differ: {key}'
    shuffled = np.random.default_rng(42).permutation(canonical)
    fit_ids = set(shuffled[:250])
    write_json(CALIBRATION_ROOT / 'partition.json', {
        'seed': 42, 'fit_ids': sorted(fit_ids), 'holdout_ids': sorted(set(canonical) - fit_ids),
        'limitation': 'Validation previously selected the checkpoint; holdout results are exploratory.'})
    summaries = []
    for (name, mode), frame in frames.items():
        folder = CALIBRATION_ROOT / name / mode
        fit = frame[frame.question_id.astype(str).isin(fit_ids)].copy()
        holdout = frame[~frame.question_id.astype(str).isin(fit_ids)].copy()
        artifact = fit_artifact(fit)
        write_json(folder / 'temperature.json', artifact)
        fit.to_csv(folder / 'calibration_fit_predictions.csv', index=False)
        holdout.to_csv(folder / 'holdout_raw_predictions.csv', index=False)
        rows = report(holdout, apply_artifact(holdout, artifact), folder / 'holdout')
        for row in rows:
            summaries.append({'model': name, 'mode': mode, 'split': 'validation_holdout',
                              'temperature': artifact['temperature'], **row})
        test_path = INPUTS[(name, mode)][1]
        if test_path is not None:
            test = read_frame(test_path)
            rows = report(test, apply_artifact(test, artifact), folder / 'test')
            for row in rows:
                summaries.append({'model': name, 'mode': mode, 'split': 'test',
                                  'temperature': artifact['temperature'], **row})
        else:
            print('Test predictions unavailable:', name, mode)
    pd.DataFrame(summaries).to_csv(CALIBRATION_ROOT / 'comparison.csv', index=False)
else:
    print('Temperature fitting is disabled.')

Test predictions unavailable: step_000208 direct


In [7]:
import pandas as pd
from IPython.display import display
comparison = CALIBRATION_ROOT / 'comparison.csv'
if comparison.is_file():
    display(pd.read_csv(comparison)[['model', 'mode', 'split', 'variant', 'temperature',
                                    'accuracy', 'mean_confidence', 'ece', 'nll', 'brier',
                                    'hc_wrong_rate', 'hc_coverage']])
for name in ['base', DPO_NAME]:
    summary = EVAL_ROOT / name / 'summary.json'
    if summary.is_file():
        results = json.loads(summary.read_text())
        display(pd.DataFrame([{'model': name, 'split': split, 'mode': mode, **metrics}
                              for split, modes in results.items() for mode, metrics in modes.items()]))

,model,mode,split,variant,temperature,accuracy,mean_confidence,ece,nll,brier,hc_wrong_rate,hc_coverage
0,base,direct,validation_holdout,raw,11.539885,0.512000,0.945559,0.442925,4.554116,0.914634,0.38400,0.836000
1,base,direct,validation_holdout,temperature,11.539885,0.512000,0.476966,0.082101,1.157291,0.619516,0.00000,0.000000
2,base,direct,test,raw,11.539885,0.468185,0.955671,0.487485,5.638082,1.000214,0.43912,0.870385
3,base,direct,test,temperature,11.539885,0.468185,0.492837,0.040658,1.216833,0.655937,0.00000,0.000000
4,step_000208,direct,validation_holdout,raw,11.612985,0.504000,0.945266,0.446214,4.595846,0.917549,0.39600,0.848000
5,step_000208,direct,validation_holdout,temperature,11.612985,0.504000,0.477998,0.067685,1.155922,0.618551,0.00000,0.000000
